In [26]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [27]:
class TrackedObject:
  def __init__(self, **kwargs):
    # history сам добавляем через super().__setattr__, чтобы не ловить в логе
    super().__setattr__('history', [])
    for name, value in kwargs.items():
      setattr(self, name, value)

  def __setattr__(self, name, value):
    if name == "history":
      super().__setattr__(name, value)
      return
    
    d = object.__getattribute__(self, "__dict__")
    before_size = sys.getsizeof(d)
    existed_before = name in d

    super().__setattr__(name, value)

    after_dict = object.__getattribute__(self, "__dict__")
    after_size = sys.getsizeof(after_dict)

    object.__getattribute__(self, "history").append({
      "action": "set",
      "name": name,
      "value": repr(value),
      "existed_before": existed_before,
      "dict_size_before": before_size,
      "dict_size_after": after_size,
      "keys_after": list(after_dict.keys()),
    })

  def __delattr__(self, name):
    if name == "history":
      super().__delattr__(name)
      return

    before_dict = object.__getattribute__(self, "__dict__")
    before_size = sys.getsizeof(before_dict)
    old_value = object.__getattribute__(self, name)

    super().__delattr__(name)

    after_dict = object.__getattribute__(self, "__dict__")
    after_size = sys.getsizeof(after_dict)

    object.__getattribute__(self, "history").append({
      "action": "delete",
      "name": name,
      "value": repr(old_value),
      "dict_size_before": before_size,
      "dict_size_after": after_size,
      "keys_after": list(after_dict.keys()),
    })

obj = TrackedObject(x=1, y=2)
obj.z = 3                # добавление нового атрибута
obj.__str__ = lambda s: "patched"  # перезатирка метода
del obj.x

print(obj.__dict__)
for event in obj.history:
  print(event)

{'history': [{'action': 'set', 'name': 'x', 'value': '1', 'existed_before': False, 'dict_size_before': 296, 'dict_size_after': 296, 'keys_after': ['history', 'x']}, {'action': 'set', 'name': 'y', 'value': '2', 'existed_before': False, 'dict_size_before': 296, 'dict_size_after': 296, 'keys_after': ['history', 'x', 'y']}, {'action': 'set', 'name': 'z', 'value': '3', 'existed_before': False, 'dict_size_before': 296, 'dict_size_after': 296, 'keys_after': ['history', 'x', 'y', 'z']}, {'action': 'set', 'name': '__str__', 'value': '<function <lambda> at 0x000001BE70DC96C0>', 'existed_before': False, 'dict_size_before': 296, 'dict_size_after': 296, 'keys_after': ['history', 'x', 'y', 'z', '__str__']}, {'action': 'delete', 'name': 'x', 'value': '1', 'dict_size_before': 296, 'dict_size_after': 296, 'keys_after': ['history', 'y', 'z', '__str__']}], 'y': 2, 'z': 3, '__str__': <function <lambda> at 0x000001BE70DC96C0>}
{'action': 'set', 'name': 'x', 'value': '1', 'existed_before': False, 'dict_size

## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [35]:
def merge(seqs):
    seqs = [list(seq) for seq in seqs if seq]
    result = []

    while seqs:
        for seq in seqs:
            candidate = seq[0]
            if not any(candidate in other[1:] for other in seqs):
                break
        else:
            raise TypeError("Inconsistent hierarchy: C3 merge failed")

        result.append(candidate)

        new_seqs = []
        for seq in seqs:
            if seq and seq[0] == candidate:
                seq.pop(0)
            if seq:
                new_seqs.append(seq)
        seqs = new_seqs

    return result

def c3_linearize(cls):
    if cls is object:
        return [object]

    bases = list(cls.__bases__)
    parent_mros = [c3_linearize(base) for base in bases]
    return [cls] + merge(parent_mros + [bases])


class O:
    def who(self):
        return "O"

class A(O):
    def who(self):
        return "A"

class B(O):
    def who(self):
        return "B"

class C(A, B):
    pass

class D(A, B):
    pass

class E(C, D):
    pass

for cls in (C, D, E):
    print(f"{cls.__name__} custom MRO:", [c.__name__ for c in c3_linearize(cls)])
    print(f"{cls.__name__} python MRO:", [c.__name__ for c in cls.__mro__])
    print()

e = E()
print(A().who())
print("e.who():", e.who())


C custom MRO: ['C', 'A', 'B', 'O', 'object']
C python MRO: ['C', 'A', 'B', 'O', 'object']

D custom MRO: ['D', 'A', 'B', 'O', 'object']
D python MRO: ['D', 'A', 'B', 'O', 'object']

E custom MRO: ['E', 'C', 'D', 'A', 'B', 'O', 'object']
E python MRO: ['E', 'C', 'D', 'A', 'B', 'O', 'object']

A
e.who(): A


Для класса C и D MRO выглядят как C-A-B-O или D-A-B-O соответственно ввиду того, что в списке наследования классы A и B в порядке A, B в обоих классах наследниках. Для класса E MRO выглядит как E-C-D-A-B-O, что объясняется тем же порядком наследуемых классов   

## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [29]:
class SecureBase:
    def __init__(self):
        self.__secret_value = "base secret"
        self._semi_private = "base semi-private"
        self.public = "base public"

class SecureChild(SecureBase):
    def __init__(self):
        super().__init__()
        self.__secret_value = "child secret"
        self._semi_private = "child semi-private"

    def dump_dict(self):
        return self.__dict__


base = SecureBase()
child = SecureChild()

print("Base __dict__:", base.__dict__)
print("Child __dict__:", child.dump_dict())

print("Base dir():", [name for name in dir(base) if "secret" in name or "semi" in name or name == "public"])
print("Child dir():", [name for name in dir(child) if "secret" in name or "semi" in name or name == "public"])

print("base._SecureBase__secret_value =", base._SecureBase__secret_value)
print("child._SecureBase__secret_value =", child._SecureBase__secret_value)
print("child._SecureChild__secret_value =", child._SecureChild__secret_value)
print("child._semi_private =", child._semi_private)

try:
    print(child.__secret_value)
except AttributeError as e:
    print("Direct access failed:", e)

Base __dict__: {'_SecureBase__secret_value': 'base secret', '_semi_private': 'base semi-private', 'public': 'base public'}
Child __dict__: {'_SecureBase__secret_value': 'base secret', '_semi_private': 'child semi-private', 'public': 'base public', '_SecureChild__secret_value': 'child secret'}
Base dir(): ['_SecureBase__secret_value', '_semi_private', 'public']
Child dir(): ['_SecureBase__secret_value', '_SecureChild__secret_value', '_semi_private', 'public']
base._SecureBase__secret_value = base secret
child._SecureBase__secret_value = base secret
child._SecureChild__secret_value = child secret
child._semi_private = child semi-private
Direct access failed: 'SecureChild' object has no attribute '__secret_value'


## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [30]:
class NoSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlotsWeak:
    __slots__ = ("x", "__weakref__")

    def __init__(self, x):
        self.x = x

def describe_instance(obj):
    d = {
        "type": type(obj).__name__,
        "has_dict": hasattr(obj, "__dict__"),
        "has_weakref": hasattr(obj, "__weakref__"),
        "size_obj": sys.getsizeof(obj),
    }
    if hasattr(obj, "__dict__"):
        d["size_dict"] = sys.getsizeof(obj.__dict__)
        d["dict_keys"] = list(obj.__dict__.keys())
    else:
        d["size_dict"] = None
        d["dict_keys"] = None
    return d

n = NoSlots(1, 2)
w = WithSlots(1, 2)
ww = WithSlotsWeak(1)

# Попытка динамического добавления атрибутов
for obj in (n, w, ww):
    try:
        obj.z = 100
        print(type(obj).__name__, "can add z -> yes")
    except AttributeError as e:
        print(type(obj).__name__, "can add z -> no:", e)

print()

for obj in (n, w, ww):
    print(describe_instance(obj))

print()

# weakref
for obj in (n, w, ww):
    try:
        r = weakref.ref(obj)
        print(type(obj).__name__, "weakref -> OK,", r())
    except TypeError as e:
        print(type(obj).__name__, "weakref -> FAIL:", e)

NoSlots can add z -> yes
WithSlots can add z -> no: 'WithSlots' object has no attribute 'z'
WithSlotsWeak can add z -> no: 'WithSlotsWeak' object has no attribute 'z'

{'type': 'NoSlots', 'has_dict': True, 'has_weakref': True, 'size_obj': 48, 'size_dict': 296, 'dict_keys': ['x', 'y', 'z']}
{'type': 'WithSlots', 'has_dict': False, 'has_weakref': False, 'size_obj': 48, 'size_dict': None, 'dict_keys': None}
{'type': 'WithSlotsWeak', 'has_dict': False, 'has_weakref': True, 'size_obj': 56, 'size_dict': None, 'dict_keys': None}

NoSlots weakref -> OK, <__main__.NoSlots object at 0x000001BE7251B5F0>
WithSlots weakref -> FAIL: cannot create weak reference to 'WithSlots' object
WithSlotsWeak weakref -> OK, <__main__.WithSlotsWeak object at 0x000001BE72B61710>


## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [33]:
class Node:
    def __init__(self, name, parent=None, weak_parent=False):
        self.name = name
        self._weak_parent = weak_parent

        if parent is None:
            self.parent = None
        elif weak_parent:
            self.parent = weakref.ref(parent)
        else:
            self.parent = parent

    def get_parent(self):
        if self.parent is None:
            return None
        if self._weak_parent:
            return self.parent()
        return self.parent



def refcount(obj):
    return sys.getrefcount(obj) - 1


# Цикл со сильными ссылками
a = Node("a")
b = Node("b", parent=a)
a.parent = b

print("Strong cycle refcounts:", refcount(a), refcount(b))

del a, b
gc.collect()   # объекты должны быть собраны как циклический мусор


# Цикл с weakref
c = Node("c")
d = Node("d", parent=c, weak_parent=True)
c.parent = d

print("Weak cycle refcounts:", refcount(c), refcount(d))

wr_c = weakref.ref(c)
print("wr_c before:", wr_c())
print("d.get_parent() before:", d.get_parent())

del c
gc.collect()

print("wr_c after deletion:", wr_c())
print("d.get_parent() after:", d.get_parent())

Strong cycle refcounts: 3 3
Weak cycle refcounts: 2 3
wr_c before: <__main__.Node object at 0x000001BE7251AA50>
d.get_parent() before: <__main__.Node object at 0x000001BE7251AA50>
wr_c after deletion: None
d.get_parent() after: None
